# Proof of Concept — IoT Network Anomaly Detection with an Autoencoder

**Dataset:** Edge-IIoTset (`DNN-EdgeIIoT-dataset.csv`)

**Core idea.** The autoencoder is trained on *normal* traffic only. It learns to
compress and rebuild normal behaviour accurately. When an attack record arrives,
the model has never seen anything like it, so it rebuilds it badly. That rebuild
failure — the **reconstruction error** — is the detection signal.

---

### How to run

Run the cells top to bottom. Each section explains what it does and why.


## 0 — Configuration

Adjust `SAMPLE_ROWS` to trade accuracy against memory. Free Colab has roughly
12 GB of RAM, so 400,000 rows is a safe starting point.


In [ ]:
# Rows to read. Set to None for the full file (needs a lot of RAM).
SAMPLE_ROWS = 400_000

# Fraction of normal traffic held out for testing (the rest trains the model)
TEST_SPLIT = 0.2

# Anything with an error above this percentile of normal errors is flagged
THRESHOLD_PCT = 95

EPOCHS     = 20
BATCH_SIZE = 256
SEED       = 42

# Path to the zip in your Google Drive — edit if you put it elsewhere
ZIP_PATH = '/content/drive/MyDrive/DNN-EdgeIIoT-dataset.csv.zip'

import numpy as np, random
np.random.seed(SEED); random.seed(SEED)


## 1 — Mount Drive and extract the dataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, glob, zipfile

assert os.path.exists(ZIP_PATH), f'File not found at: {ZIP_PATH}'
print('Archive size:', round(os.path.getsize(ZIP_PATH)/1e6), 'MB')

os.makedirs('/content/data', exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    print('Archive contents:', z.namelist())
    z.extractall('/content/data')

csvs = glob.glob('/content/data/**/*.csv', recursive=True)
CSV_PATH = max(csvs, key=os.path.getsize)   # largest CSV is the dataset
print('Using:', CSV_PATH, '|', round(os.path.getsize(CSV_PATH)/1e6), 'MB')


## 2 — Load and inspect


In [ ]:
import pandas as pd

# The CSV is ordered — normal traffic first, attacks after. Reading the first N
# rows would return normal traffic only, so we sample evenly across the whole file
# by walking it in chunks and taking a fraction from each.

CHUNK = 100_000

total_rows = sum(1 for _ in open(CSV_PATH)) - 1
print(f'Rows in file: {total_rows:,}')

if SAMPLE_ROWS is None or SAMPLE_ROWS >= total_rows:
    df = pd.read_csv(CSV_PATH, low_memory=False)
else:
    frac  = SAMPLE_ROWS / total_rows
    parts = []
    for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNK, low_memory=False):
        parts.append(chunk.sample(frac=frac, random_state=SEED))
    df = pd.concat(parts, ignore_index=True)
    del parts

print('Shape:', df.shape)
df.head()


In [ ]:
# Locate the label columns
label_cols = [c for c in df.columns if 'attack' in c.lower() or 'label' in c.lower()]
print('Label columns found:', label_cols)

LABEL_COL = 'Attack_label' if 'Attack_label' in df.columns else label_cols[0]
TYPE_COL  = 'Attack_type'  if 'Attack_type'  in df.columns else None

print()
print(df[LABEL_COL].value_counts().rename({0: 'Normal', 1: 'Attack'}))
if TYPE_COL:
    print()
    print(df[TYPE_COL].value_counts())

# Guard: both classes must be present or everything downstream fails
counts = df[LABEL_COL].value_counts()
assert 0 in counts.index and 1 in counts.index, (
    'The sample contains only one class. Raise SAMPLE_ROWS and re-run.')
print('\nBoth classes present — safe to continue.')


## 3 — Clean the feature set

Three groups of columns are removed:

| Group | Examples | Why it is dropped |
|---|---|---|
| **Identifiers** | `ip.src_host`, `frame.time` | Free-text values tied to one capture; they do not generalise to another network |
| **Labels** | `Attack_label`, `Attack_type` | These *are* the answer — leaving them in would let the model cheat |
| **Constants** | any column with a single value | Carries no information |


In [ ]:
import numpy as np

# Identifier and free-text columns that do not generalise
ID_LIKE = ['frame.time', 'ip.src_host', 'ip.dst_host',
           'arp.src.proto_ipv4', 'arp.dst.proto_ipv4',
           'http.file_data', 'http.request.full_uri', 'http.request.version',
           'icmp.transmit_timestamp', 'tcp.options', 'tcp.payload',
           'mqtt.msg', 'dns.qry.name', 'http.referer', 'mqtt.topic']

drop_cols  = [c for c in ID_LIKE if c in df.columns]
drop_cols += [c for c in [LABEL_COL, TYPE_COL] if c and c in df.columns]

X_all = df.drop(columns=drop_cols)

# Keep numeric features only
X_all = X_all.select_dtypes(include=[np.number])

# Replace infinities and missing values
X_all = X_all.replace([np.inf, -np.inf], np.nan).fillna(0)

# Drop constant columns
const_cols = [c for c in X_all.columns if X_all[c].nunique() <= 1]
X_all = X_all.drop(columns=const_cols)

print(f'Dropped {len(drop_cols)} identifier/label columns')
print(f'Dropped {len(const_cols)} constant columns')
print(f'Remaining numeric features: {X_all.shape[1]}')
print()
print('Features:', list(X_all.columns))


## 4 — Split normal and attack traffic

The model trains on a *portion* of normal traffic. The held-out normal records are
used to set the threshold and to measure the false alarm rate.


In [ ]:
from sklearn.model_selection import train_test_split

y = df[LABEL_COL].values
X_normal = X_all[y == 0]
X_attack = X_all[y == 1]

X_tr, X_te_norm = train_test_split(X_normal, test_size=TEST_SPLIT, random_state=SEED)

print(f'Train (normal only) : {X_tr.shape[0]:,}')
print(f'Test  normal        : {X_te_norm.shape[0]:,}')
print(f'Test  attack        : {X_attack.shape[0]:,}')


## 5 — Scale the features

> ⚠ **The scaler is fitted on training data only** (`fit_transform`).
> Test data is only transformed (`transform`, never `fit`). Fitting on test data
> would let information about the attacks leak into preprocessing — **data leakage**,
> which makes the reported result meaningless.

> ⚠ **Second pitfall: do not clip the scaled values** with `np.clip(X, 0, 1)`.
> The scaler maps normal traffic into the 0–1 range. Attack records fall *outside*
> that range (e.g. 4.7), and that is exactly what makes them detectable.
> Clipping them back to 1 erases the signal — in testing, detection dropped from
> ~99% to ~3%.


In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
Xtr_s  = scaler.fit_transform(X_tr)      # fit on training data only
Xten_s = scaler.transform(X_te_norm)     # transform only
Xatk_s = scaler.transform(X_attack)      # transform only

print('Train range  :', round(Xtr_s.min(), 3), '->', round(Xtr_s.max(), 3))
print('Attack range :', round(Xatk_s.min(), 3), '->', round(Xatk_s.max(), 3))
print('Features     :', Xtr_s.shape[1])

out_of_range = ((Xatk_s < 0) | (Xatk_s > 1)).mean()
print(f'\nAttack values outside the normal range: {out_of_range:.2%}')
print('A high value here is a good sign — attacks differ measurably from normal.')


## 6 — Build the autoencoder

The input is compressed to half its width, then to a quarter, then expanded back.
The bottleneck forces the network to learn *patterns* rather than memorise rows.

**Architecture:** `n → n/2 → n/4 → n/2 → n`

This follows the standard autoencoder design used for network anomaly detection;
the layer widths adapt to however many features survive cleaning.


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

tf.random.set_seed(SEED)

n_feat = Xtr_s.shape[1]
h1 = max(32, n_feat // 2)
h2 = max(16, n_feat // 4)

inp = Input(shape=(n_feat,))
e   = Dense(h1, activation='relu')(inp)   # encoder
e   = Dense(h2, activation='relu')(e)     # bottleneck
d   = Dense(h1, activation='relu')(e)     # decoder
out = Dense(n_feat, activation='sigmoid')(d)

ae = Model(inp, out)
ae.compile(optimizer=Adam(1e-3), loss='mse')

print(f'Architecture: {n_feat} -> {h1} -> {h2} -> {h1} -> {n_feat}')
ae.summary()


## 7 — Train

Note that the input and the target are the same array. The model's only job is to
reproduce what it is given.


In [ ]:
hist = ae.fit(
    Xtr_s, Xtr_s,            # input == target
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    shuffle=True,
    verbose=1
)


In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 120

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(hist.history['loss'],     label='Train',      lw=2)
ax.plot(hist.history['val_loss'], label='Validation', lw=2, ls='--')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE loss')
ax.set_title('Autoencoder training')
ax.legend(); ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig('/content/fig1_training.png', dpi=150)
plt.show()


## 8 — Set the detection threshold

The threshold is the 95th percentile of reconstruction error on *training* data.
By construction this accepts a **5% false alarm rate** on normal traffic — the
price paid for catching more attacks.


In [ ]:
def recon_error(model, X):
    """Mean squared difference between each row and its reconstruction."""
    R = model.predict(X, batch_size=1024, verbose=0)
    return np.mean(np.square(X - R), axis=1)

err_train = recon_error(ae, Xtr_s)
err_norm  = recon_error(ae, Xten_s)
err_atk   = recon_error(ae, Xatk_s)

threshold = np.percentile(err_train, THRESHOLD_PCT)

print(f'Threshold (p{THRESHOLD_PCT} of training error) = {threshold:.6f}')
print(f'Mean error, normal : {err_norm.mean():.6f}')
print(f'Mean error, attack : {err_atk.mean():.6f}')
print(f'Ratio              : {err_atk.mean()/max(err_norm.mean(), 1e-12):.1f}x')


## 9 — Results

These are the numbers to put on the slide.


In [ ]:
tp = int((err_atk  > threshold).sum())   # attack correctly flagged
fn = int((err_atk <= threshold).sum())   # attack missed
fp = int((err_norm > threshold).sum())   # false alarm
tn = int((err_norm<= threshold).sum())   # normal correctly passed

detection = tp / (tp + fn)
false_pos = fp / (fp + tn)
precision = tp / max(tp + fp, 1)
f1        = 2*precision*detection / max(precision + detection, 1e-12)
accuracy  = (tp + tn) / (tp + tn + fp + fn)

print('=' * 44)
print(f"{'Attack samples':<24}{tp+fn:>12,}")
print(f"{'Detected':<24}{tp:>12,}")
print(f"{'Missed':<24}{fn:>12,}")
print('-' * 44)
print(f"{'Detection rate':<24}{detection:>11.2%}")
print(f"{'False positive rate':<24}{false_pos:>11.2%}")
print(f"{'Precision':<24}{precision:>11.2%}")
print(f"{'F1 score':<24}{f1:>11.2%}")
print(f"{'Accuracy':<24}{accuracy:>11.2%}")
print('=' * 44)


### Key figure — normal vs attack error distribution

This is the single most persuasive chart in the presentation.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

lo = min(err_norm.min(), err_atk.min())
hi = np.percentile(np.concatenate([err_norm, err_atk]), 99.5)
bins = np.linspace(lo, hi, 70)

ax.hist(err_norm, bins=bins, alpha=.65, label='Normal', color='#2a9d8f', density=True)
ax.hist(err_atk,  bins=bins, alpha=.65, label='Attack', color='#e76f51', density=True)
ax.axvline(threshold, color='#264653', ls='--', lw=2,
           label=f'Threshold (p{THRESHOLD_PCT})')

ax.set_xlabel('Reconstruction error'); ax.set_ylabel('Density')
ax.set_title(f'Normal vs attack — detection rate {detection:.1%}')
ax.legend(); ax.grid(alpha=.25)
fig.tight_layout()
fig.savefig('/content/fig2_separation.png', dpi=150)
plt.show()


## 10 — Detection rate per attack type

Shows where the model is strong and where it is weak. Useful material for the
defence, and honest about the method's limits.


In [ ]:
if TYPE_COL:
    types = df.loc[y == 1, TYPE_COL].values
    rows = []
    for t in pd.unique(types):
        m = types == t
        rows.append({'Attack type': t,
                     'Samples' : int(m.sum()),
                     'Detected': int((err_atk[m] > threshold).sum()),
                     'Rate'    : (err_atk[m] > threshold).mean()})
    per_type = pd.DataFrame(rows).sort_values('Rate', ascending=False)
    per_type['Rate'] = (per_type['Rate'] * 100).round(2)
    display(per_type.reset_index(drop=True))
else:
    print('No attack-type column in this dataset')


In [ ]:
if TYPE_COL:
    d = per_type.sort_values('Rate')
    fig, ax = plt.subplots(figsize=(7, .38*len(d) + 1.6))
    ax.barh(d['Attack type'], d['Rate'], color='#e76f51')
    ax.set_xlabel('Detection rate (%)'); ax.set_xlim(0, 100)
    ax.set_title('Detection rate per attack type')
    ax.grid(axis='x', alpha=.25)
    fig.tight_layout()
    fig.savefig('/content/fig3_per_attack.png', dpi=150)
    plt.show()


## 11 — Baseline comparison

This section is not optional. The examining committee will ask why a deep model is
needed at all, and the answer has to be backed by a number.


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM

# Subsample — One-Class SVM scales poorly on large data
n_fit = min(30_000, len(Xtr_s))
idx   = np.random.choice(len(Xtr_s), n_fit, replace=False)
fit_sample = Xtr_s[idx]

print('Fitting Isolation Forest...')
iso = IsolationForest(n_estimators=100, contamination=0.05,
                      random_state=SEED, n_jobs=-1)
iso.fit(fit_sample)

print('Fitting One-Class SVM...')
svm = OneClassSVM(kernel='rbf', nu=0.05, gamma='scale')
svm.fit(fit_sample[:10_000])
print('Done')


In [ ]:
def rates(pred_atk, pred_norm):
    """scikit-learn returns -1 for anomalies."""
    return (pred_atk == -1).mean(), (pred_norm == -1).mean()

iso_d, iso_f = rates(iso.predict(Xatk_s), iso.predict(Xten_s))
svm_d, svm_f = rates(svm.predict(Xatk_s), svm.predict(Xten_s))

comp = pd.DataFrame([
    {'Method': 'Autoencoder',      'Detection %': detection*100, 'False positive %': false_pos*100},
    {'Method': 'Isolation Forest', 'Detection %': iso_d*100,     'False positive %': iso_f*100},
    {'Method': 'One-Class SVM',    'Detection %': svm_d*100,     'False positive %': svm_f*100},
]).round(2)

display(comp)


> **If Isolation Forest scores higher, that is expected — and here is the answer:**
>
> *Isolation Forest performs well on blatant attacks such as DDoS and scanning,
> where a single record is already statistically extreme. But it evaluates each
> record independently — it has no notion of time.*
>
> *The attacks this project targets — sensor reading manipulation and replay —
> look perfectly normal in any single record. What exposes them is the **sequence**
> over time, which requires an LSTM and is absent from Isolation Forest.*

Also compare the **false positive** column. A method that detects more sometimes
does so by raising far more false alarms, which matters in a real deployment.


## 12 — Save artefacts to Drive


In [ ]:
import shutil, json

OUT = '/content/drive/MyDrive/PoC_results'
os.makedirs(OUT, exist_ok=True)

for f in ['fig1_training.png', 'fig2_separation.png', 'fig3_per_attack.png']:
    p = '/content/' + f
    if os.path.exists(p):
        shutil.copy(p, OUT)

summary = {
    'dataset'            : os.path.basename(CSV_PATH),
    'rows_loaded'        : int(len(df)),
    'features_used'      : int(n_feat),
    'train_normal'       : int(len(X_tr)),
    'test_normal'        : int(len(X_te_norm)),
    'test_attack'        : int(len(X_attack)),
    'threshold'          : float(threshold),
    'detection_rate'     : float(detection),
    'false_positive_rate': float(false_pos),
    'precision'          : float(precision),
    'f1'                 : float(f1),
    'accuracy'           : float(accuracy),
}

with open(f'{OUT}/summary.json', 'w') as fh:
    json.dump(summary, fh, indent=2)

comp.to_csv(f'{OUT}/comparison.csv', index=False)
if TYPE_COL:
    per_type.to_csv(f'{OUT}/per_attack_type.csv', index=False)

ae.save(f'{OUT}/autoencoder.keras')

print('Saved to:', OUT)
print(json.dumps(summary, indent=2))


---

## Done

`PoC_results` in your Drive now contains:

| File | Contents |
|---|---|
| `fig1_training.png` | Training and validation loss curves |
| `fig2_separation.png` | **Key figure** — normal vs attack error distribution |
| `fig3_per_attack.png` | Detection rate broken down by attack type |
| `comparison.csv` | Autoencoder vs Isolation Forest vs One-Class SVM |
| `summary.json` | All metrics |
| `autoencoder.keras` | The trained model |

**Before the defence:** run the notebook end to end once more from a fresh runtime
to confirm the results are stable.

---

### Reference

Ferrag, M. A., Friha, O., Hamouda, D., Maglaras, L., Janicke, H. (2022).
*Edge-IIoTset: A New Comprehensive Realistic Cyber Security Dataset of IoT and
IIoT Applications for Centralized and Federated Learning.* IEEE Access.
DOI: 10.1109/ACCESS.2022.3165809
